# Stage 3 DPD Safe-Zone — Phase B: notebook setup

**ПРОСТЫМ ЯЗЫКОМ:** Загружаем 4 сырых CSV из SQL-выгрузки (`stage3_safezone_rolling_extract.sql`), строим для каждого из 12 отчётных месяцев 6-месячное окно DPD по займу, и считаем `restr_active_pct` — долю месяцев в этом окне, покрытых активной реструктуризацией (`grace_od_*`/`grace_int_*` из `restructuring_v2`, без учёта отменённых по `canc_date`). Это подготовка данных для Фазы C (симуляция порога) — сама симуляция здесь ещё не реализована.

See [`docs/analysis/stage3_safezone_plan.md`](../docs/analysis/stage3_safezone_plan.md) for the full plan (Phases A–E).

**Before running:** point `RAW_DATA_DIR` below at your exported CSVs and confirm the filenames in `FILES` match what you actually exported — the diagnostic cell right after lists what's actually in the folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DATA_DIR = Path(r"C:\project_mz\surau\DPDRelaxing\raw_data")

# Adjust these to your actual exported filenames if they differ.
FILES = {
    "report_dates": "report_dates.csv",
    "stage3_pool": "stage3_pool.csv",
    "dpd_panel": "dpd_panel.csv",
    "restructuring_events": "restructuring_events.csv",
}

LAST_ASOF = pd.Timestamp("2026-07-01")
LOOKBACK_MONTHS = 6

In [ ]:
print("CSV files found in RAW_DATA_DIR:")
for f in sorted(RAW_DATA_DIR.glob("*.csv")):
    print(" ", f.name)

## Load raw extracts

Matches the four result sets from `sql/stage3_safezone_rolling_extract.sql` §0/§1/§2/§3 — no transformation, just typed columns.

In [ ]:
report_dates = pd.read_csv(RAW_DATA_DIR / FILES["report_dates"], parse_dates=["asof_date"])

stage3_pool = pd.read_csv(RAW_DATA_DIR / FILES["stage3_pool"], parse_dates=["date", "portfolio_asof"])

dpd_panel = pd.read_csv(RAW_DATA_DIR / FILES["dpd_panel"], parse_dates=["snap_date"])

restructuring_events = pd.read_csv(
    RAW_DATA_DIR / FILES["restructuring_events"],
    parse_dates=[
        "restructuring_date",
        "new_maturity_date",
        "canc_date",
        "grace_od_begin_date",
        "grace_od_end_date",
        "grace_int_begin_date",
        "grace_int_end_date",
        "report_date",
    ],
)

## Sanity check — row counts against what SQL Server reported (23.07.2026)

In [ ]:
expected = {
    "report_dates": 12,
    "stage3_pool": 481_818,
    "dpd_panel": 1_080_891,
    "restructuring_events": 91_679,
}
actual = {
    "report_dates": len(report_dates),
    "stage3_pool": len(stage3_pool),
    "dpd_panel": len(dpd_panel),
    "restructuring_events": len(restructuring_events),
}
for name, expected_count in expected.items():
    got = actual[name]
    flag = "OK" if got == expected_count else "CHECK — differs from the SQL-side count"
    print(f"{name:22s} expected {expected_count:>10,}  got {got:>10,}  [{flag}]")

## Per-portfolio 6-month lookback window

For a given `portfolio_asof`, slice `dpd_panel` down to the `LOOKBACK_MONTHS` ending at that date (inclusive), and label each row with a `month_offset` (0 = the portfolio's own month, negative = further back).

In [ ]:
def build_lookback_dpd(
    dpd_panel: pd.DataFrame, portfolio_asof: pd.Timestamp, lookback_months: int = LOOKBACK_MONTHS
) -> pd.DataFrame:
    """DPD/category rows for the lookback window ending at portfolio_asof (inclusive)."""
    window_start = portfolio_asof - pd.DateOffset(months=lookback_months - 1)
    window = dpd_panel[
        (dpd_panel["snap_date"] >= window_start) & (dpd_panel["snap_date"] <= portfolio_asof)
    ].copy()
    window["month_offset"] = (
        (window["snap_date"].dt.year - portfolio_asof.year) * 12
        + (window["snap_date"].dt.month - portfolio_asof.month)
    )
    return window

## `restr_active_pct` — flag months covered by a live (non-cancelled) grace period

A month counts as restructuring-active if, as of that snapshot, there is a restructuring event that (a) has already started (`restructuring_date <= snap_date`), (b) has not been cancelled by then (`canc_date` null or later than `snap_date`), and (c) the snapshot date falls inside either the principal grace window (`grace_od_begin_date`–`grace_od_end_date`) or the interest grace window (`grace_int_begin_date`–`grace_int_end_date`). A loan can match more than one event — `any()` across all qualifying events, not just the most recent, since an older event's grace window can still be running.

In [ ]:
def flag_restructuring_active(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """One row per (contract_number, snap_date) with restr_active = True/False."""
    events = restructuring_events.rename(columns={"loan_id": "contract_number"})
    merged = dpd_window.merge(events, on="contract_number", how="left")

    event_started = merged["restructuring_date"] <= merged["snap_date"]
    not_cancelled = merged["canc_date"].isna() | (merged["canc_date"] > merged["snap_date"])
    in_od_grace = (
        merged["grace_od_begin_date"].notna()
        & merged["grace_od_end_date"].notna()
        & (merged["snap_date"] >= merged["grace_od_begin_date"])
        & (merged["snap_date"] <= merged["grace_od_end_date"])
    )
    in_int_grace = (
        merged["grace_int_begin_date"].notna()
        & merged["grace_int_end_date"].notna()
        & (merged["snap_date"] >= merged["grace_int_begin_date"])
        & (merged["snap_date"] <= merged["grace_int_end_date"])
    )
    merged["restr_active"] = event_started & not_cancelled & (in_od_grace | in_int_grace)

    return (
        merged.groupby(["contract_number", "snap_date"])["restr_active"]
        .any()
        .reset_index()
    )


def compute_restr_active_pct(
    dpd_window: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    """Share of the lookback window's months that were restructuring-active, per contract."""
    flags = flag_restructuring_active(dpd_window, restructuring_events)
    return (
        flags.groupby("contract_number")["restr_active"]
        .mean()
        .rename("restr_active_pct")
        .reset_index()
    )

## Build `restr_active_pct` for all 12 portfolio months

In [ ]:
restr_active_pct_by_portfolio = {}

for _, row in report_dates.iterrows():
    label = row["portfolio_label"]
    asof = row["asof_date"]
    window = build_lookback_dpd(dpd_panel, asof)
    restr_active_pct_by_portfolio[label] = compute_restr_active_pct(
        window, restructuring_events
    ).set_index("contract_number")["restr_active_pct"]

restr_active_pct = pd.concat(restr_active_pct_by_portfolio, axis=1)
restr_active_pct.head()

## Transparency metric — % of the Stage 3 population with a restructuring event on record

Per Phase B of the plan: report how much of the pool even *has* a restructuring event, per portfolio month — separate from `restr_active_pct`, which only covers loans that do.

In [ ]:
def restructuring_coverage_summary(
    stage3_pool: pd.DataFrame, restructuring_events: pd.DataFrame
) -> pd.DataFrame:
    contracts_with_event = set(restructuring_events["loan_id"].unique())
    tagged = stage3_pool.assign(
        has_restructuring_event=lambda d: d["contract_number"].isin(contracts_with_event)
    )
    summary = tagged.groupby("portfolio_label")["has_restructuring_event"].agg(
        with_event="sum", total_loans="count"
    )
    summary["pct_with_event"] = (summary["with_event"] / summary["total_loans"] * 100).round(1)
    return summary.sort_index()


coverage = restructuring_coverage_summary(stage3_pool, restructuring_events)
coverage

## Next: Phase C (not yet built here)

This notebook covers Phase B of `stage3_safezone_plan.md` — loading the raw extracts, building the 6-month lookback windows, and `restr_active_pct` / coverage. Phase C (the threshold × re-default-rate simulation across n ∈ {0, 3, 7, ..., 30}) and Phase D (Hypothesis 1 vs. Hypothesis 2) build on top of what's here — run this first, confirm the sanity checks and coverage numbers look right, then continue.